# Grad-CAM for Workflow 3 (hybrid CNN + XGBoost)
## BRISC 2025 - visual explainability

**Catarina Bota | UCP**

This notebook produces **Grad-CAM** heatmaps showing **where the CNN looks** when classifying each MRI image.

**Technical note:** Grad-CAM requires gradients, so it applies only to the CNN backbone (not to XGBoost, which is gradient-free). Since the hybrid workflow shares the CNN backbone from Workflow 2, the Grad-CAM reveals what the CNN extracted before XGBoost classified.

**Outputs:**
- One 4-row figure (one example per class) with heatmap + overlay
- Individual PNGs saved to `workflow3_gradcam/`

**Runtime: ~10-15 min of GPU**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import cv2

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Configuration and model

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive'
MODEL_PATH = f'{DRIVE_BASE}/brisc_gui/modelo_brisc.h5'
TEST_DIR   = f'{DRIVE_BASE}/brisc2025_clean/classification_task/test'  # dataset forensically cleaned
OUTPUT_DIR = f'{DRIVE_BASE}/brisc_gui/workflow3_gradcam'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 128
CLASSES  = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

model = load_model(MODEL_PATH)
model.summary()

## 3. Identify the last convolutional layer

Grad-CAM needs the output of the **last Conv2D** (before pooling/flatten). Detect it automatically.

In [ ]:
# Find last conv layer automatically
last_conv_name = None
for layer in model.layers[::-1]:
    if 'conv' in layer.name.lower():
        last_conv_name = layer.name
        break

if last_conv_name is None:
    raise RuntimeError('No Conv2D layer found')

print(f'Last conv layer: {last_conv_name}')
print(f'Output shape: {model.get_layer(last_conv_name).output.shape}')

## 4. Grad-CAM function

Canonical implementation (Selvaraju et al., 2017): gradient of the predicted class output w.r.t. the last conv layer's feature maps, weighted by the global-average-pooled gradients.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_name, pred_index=None):
    """Generate a Grad-CAM heatmap.

    img_array  : np.array of shape (1, H, W, 3), normalized to [0, 1]
    pred_index : if None, uses the predicted class
    """
    grad_model = Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index.numpy() if hasattr(pred_index, 'numpy') else pred_index)

print('Grad-CAM function ready.')

## 5. Overlay function

Overlays the heatmap (jet colormap) on the original image.

In [ ]:
def overlay_heatmap(img, heatmap, alpha=0.45, cmap='jet'):
    """Overlay colored heatmap on the original image.

    img     : array (H, W, 3) in [0, 1]
    heatmap : array (h, w) in [0, 1], usually smaller than img
    """
    H, W = img.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (W, H))
    cmap_fn = mcm.get_cmap(cmap)
    heatmap_colored = cmap_fn(heatmap_resized)[..., :3]
    blended = (1 - alpha) * img + alpha * heatmap_colored
    return np.clip(blended, 0, 1)

print('Overlay function ready.')

## 6. Pick one image per class (from the test set)

In [ ]:
examples = {}
for cls in CLASSES:
    cls_dir = os.path.join(TEST_DIR, cls)
    if not os.path.exists(cls_dir):
        print(f'Warning: {cls_dir} not found')
        continue
    files = sorted(glob.glob(os.path.join(cls_dir, '*')))
    if files:
        examples[cls] = files[0]  # first image (deterministic)

for cls, path in examples.items():
    print(f'  {cls:12s} -> {os.path.basename(path)}')

## 7. Generate heatmaps and composite figure

A 4x3 grid: row = class, columns = (original | raw heatmap | overlay).

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(9, 3*len(CLASSES)),
                          constrained_layout=True)

for row, (true_class, img_path) in enumerate(examples.items()):
    img_pil = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = img_to_array(img_pil) / 255.0
    img_batch = np.expand_dims(img_arr, axis=0)

    preds = model.predict(img_batch, verbose=0)
    pred_idx = int(np.argmax(preds[0]))
    pred_class = CLASSES[pred_idx]
    pred_prob  = preds[0, pred_idx] * 100

    heatmap, _ = make_gradcam_heatmap(img_batch, model, last_conv_name,
                                       pred_index=pred_idx)
    overlay = overlay_heatmap(img_arr, heatmap, alpha=0.45, cmap='jet')

    correct = (true_class == pred_class)
    title_color = '#1F4E4A' if correct else '#C24B4B'
    title_mark  = 'OK' if correct else 'X'

    axes[row, 0].imshow(img_arr)
    axes[row, 0].set_title(f'True: {true_class}',
                            fontsize=10, color='#333')
    axes[row, 1].imshow(heatmap, cmap='jet')
    axes[row, 1].set_title('Grad-CAM heatmap',
                            fontsize=10, color='#333')
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(
        f'[{title_mark}]  Pred: {pred_class} ({pred_prob:.1f}%)',
        fontsize=10, color=title_color, fontweight='bold')

    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('Grad-CAM: where the CNN looks (Workflow 3 backbone)',
              fontsize=13, fontweight='bold', y=1.02)
plt.savefig(f'{OUTPUT_DIR}/fig_gradcam_grid.png',
            bbox_inches='tight', dpi=180)
plt.savefig(f'{OUTPUT_DIR}/fig_gradcam_grid.pdf', bbox_inches='tight')
plt.show()
print(f'\nSaved to {OUTPUT_DIR}/fig_gradcam_grid.[png,pdf]')

## 8. (Optional) High-resolution individual heatmaps

Larger overlay versions, one per class (useful for slides).

In [ ]:
for true_class, img_path in examples.items():
    img_pil = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = img_to_array(img_pil) / 255.0
    img_batch = np.expand_dims(img_arr, axis=0)
    preds = model.predict(img_batch, verbose=0)
    pred_idx = int(np.argmax(preds[0]))
    heatmap, _ = make_gradcam_heatmap(img_batch, model, last_conv_name,
                                       pred_index=pred_idx)
    overlay = overlay_heatmap(img_arr, heatmap, alpha=0.5, cmap='jet')

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(overlay); ax.axis('off')
    pred_class = CLASSES[pred_idx]
    pred_prob  = preds[0, pred_idx] * 100
    title = f'{true_class} -> {pred_class} ({pred_prob:.1f}%)'
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.savefig(f'{OUTPUT_DIR}/gradcam_{true_class}.png',
                bbox_inches='tight', dpi=180)
    plt.show()

## 9. Discussion (to include in the article)

Inspect each row:
1. **Glioma**: does the heatmap concentrate on the tumour (hyper-intense region)?
2. **Meningioma**: does it point to the periphery (typical of extra-axial meningiomas)?
3. **No tumour**: is it diffuse (no focus) or centred?
4. **Pituitary**: does it point to the skull base (sella turcica)?

If yes -> the CNN learned **anatomically meaningful localizations**, validating the model.
If no -> the CNN may be using **background artefacts** or **acquisition bias** (important to report).

For the paper, add a new figure (Fig. 5) with the 4x3 grid and discuss each class.

## 10. Next steps

When this notebook finishes:
1. Download `fig_gradcam_grid.png` (Drive -> `workflow3_gradcam/`)
2. Copy it to `latex_artigo/figures/` in the project
3. Add it to the article (snippet ready in `snippets_para_wf4_wf5.tex`)

For Plan B (journal extension), also add:
- **SHAP** on XGBoost (`shap.TreeExplainer(xgb).shap_values(X_test)`)
- Show which of the 128 CNN features are most important to XGBoost
- Combine with Grad-CAM: "the CNN looks here -> XGBoost uses these features"